# **Pipeline de Preprocesamiento de Datos**
A continuación se mostrará el órden en el que se preprocesaron los datos en la Entrega 1

### **Importación de librerías**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import LabelEncoder
import json
pd.options.display.max_columns = None

### **Importar datos**

In [2]:
data_path = 'data/'
df_lists = []

customers_df = pd.read_csv(data_path + 'olist_customers_dataset.csv')
geolocation_df = pd.read_csv(data_path + 'olist_geolocation_dataset.csv')
order_items_df = pd.read_csv(data_path + 'olist_order_items_dataset.csv')
order_payments_df = pd.read_csv(data_path + 'olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv(data_path + 'olist_order_reviews_dataset.csv')
orders_df = pd.read_csv(data_path + 'olist_orders_dataset.csv')
products_df = pd.read_csv(data_path + 'olist_products_dataset.csv')
sellers_df = pd.read_csv(data_path + 'olist_sellers_dataset.csv')
category_translation_df = pd.read_csv(data_path + 'product_category_name_translation.csv')

df_lists.append(customers_df)
df_lists.append(geolocation_df)
df_lists.append(order_items_df)
df_lists.append(order_payments_df)
df_lists.append(order_reviews_df)
df_lists.append(orders_df)
df_lists.append(products_df)
df_lists.append(sellers_df)
df_lists.append(category_translation_df)

df_names = ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']

### **Limpieza de datos**

In [3]:
from scipy import stats


def detect_outliers(df, column, method='zscore', threshold=3):
    
    if column not in df.columns or not np.issubdtype(df[column].dtype, np.number):
        return pd.Series(False, index=df.index)
    
    if method == 'zscore':
        z_scores = np.abs(stats.zscore(df[column].dropna()))
        outliers = pd.Series(False, index=df.index)
        outliers[df[column].dropna().index] = z_scores > threshold
        return outliers
    
    elif method == 'iqr':
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        return (df[column] < lower_bound) | (df[column] > upper_bound)
    
    else:
        raise ValueError("Método no reconocido. Use 'zscore' o 'iqr'.")

def clean_dataframe(df, name):
    print(f"\n--- LIMPIEZA DE {name.upper()} ---")
    
    # Copia para no modificar el original
    df_clean = df.copy()
    changes = {}
    
    #Verificar duplicados
    n_duplicates = df_clean.duplicated().sum()
    if n_duplicates > 0:
        df_clean = df_clean.drop_duplicates()
        changes['duplicados_eliminados'] = n_duplicates
        print(f"- Se eliminaron {n_duplicates} filas duplicadas")
    
    # Convertir columnas de fechas a datetime
    date_columns = [col for col in df_clean.columns if any(date_term in col.lower() 
                                                         for date_term in ['date', 'time', '_at'])]
    for col in date_columns:
        if df_clean[col].dtype == object:
            try:
                df_clean[col] = pd.to_datetime(df_clean[col])
                changes[f'tipo_{col}'] = 'convertido a datetime'
                print(f"- Columna '{col}' convertida a datetime")
            except:
                pass
    
    # manejo de  valores extremos en columnas numéricas
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        # Solo buscar outliers en columnas que no sean IDs o códigos
        if not any(id_term in col.lower() for id_term in ['id', 'code', 'zip', 'encoded']):
            outliers = detect_outliers(df_clean, col, method='iqr')
            n_outliers = outliers.sum()
            
            if n_outliers > 0 and n_outliers < len(df_clean) * 0.05:  # Si hay menos del 5% de outliers
                #recortar (clipping)
                Q1 = df_clean[col].quantile(0.01)  # Percentil 1%
                Q3 = df_clean[col].quantile(0.99)  # Percentil 99%
                
                orig_min = df_clean[col].min()
                orig_max = df_clean[col].max()
                
                # Aplicar recorte
                df_clean[col] = df_clean[col].clip(Q1, Q3)
                
                changes[f'outliers_{col}'] = f'{n_outliers} valores recortados (min: {orig_min:.2f} → {Q1:.2f}, max: {orig_max:.2f} → {Q3:.2f})'
                print(f"- Columna '{col}': {n_outliers} outliers recortados a percentiles 1-99%")
    

### **Manejo de Valores Nulos**

In [4]:
# Manejo de valores nulos en el dataset 'products_df'
products_df['product_category_name'] = products_df['product_category_name'].fillna('unknown')
products_df['product_name_lenght'] = products_df['product_name_lenght'].fillna(products_df['product_name_lenght'].median())
products_df['product_description_lenght'] = products_df['product_description_lenght'].fillna(products_df['product_description_lenght'].median())
products_df['product_photos_qty'] = products_df['product_photos_qty'].fillna(products_df['product_photos_qty'].median())
products_df['product_weight_g'] = products_df['product_weight_g'].fillna(products_df['product_weight_g'].median())
products_df['product_length_cm'] = products_df['product_length_cm'].fillna(products_df['product_length_cm'].median())
products_df['product_height_cm'] = products_df['product_height_cm'].fillna(products_df['product_height_cm'].median())
products_df['product_width_cm'] = products_df['product_width_cm'].fillna(products_df['product_width_cm'].median())

# Manejo de valores nulos para orders
orders_df['order_status_detailed'] = orders_df['order_status']
orders_df.loc[orders_df['order_approved_at'].isnull(), 'order_status_detailed'] = 'pending_approval'
orders_df['order_approved_at'] = orders_df['order_approved_at'].fillna(pd.NaT)
orders_df['order_delivered_carrier_date'] = orders_df['order_delivered_carrier_date'].fillna(pd.NaT)
orders_df['order_delivered_customer_date'] = orders_df['order_delivered_customer_date'].fillna(pd.NaT)

# Manejo de valores nulos para order reviews
order_reviews_df['review_comment_title'] = order_reviews_df['review_comment_title'].fillna('No Title')
order_reviews_df['review_comment_message'] = order_reviews_df['review_comment_message'].fillna('No Comment')

### **Codificación de variables categóricas**

In [5]:
def apply_label_encoding(df, column, return_mapping=False):
    if column not in df.columns or df[column].isnull().all():
        if return_mapping:
            return df, None
        return df

    le = LabelEncoder()
    df_copy = df.copy()

    temp_col = df_copy[column].fillna('MISSING')
    le.fit(temp_col)
    df_copy[column] = le.transform(temp_col)

    if return_mapping:
        mapping = dict(zip(le.classes_, range(len(le.classes_))))
        return df_copy, mapping

    return df_copy


# Función  para one-hot encoding
def apply_onehot_encoding(df, column, max_categories=15):
    if column not in df.columns:
        return df

    df_copy = df.copy()
    n_unique = df_copy[column].nunique()

    if n_unique <= max_categories:
        dummies = pd.get_dummies(df_copy[column], prefix=column, dummy_na=df_copy[column].isnull().any())
        df_copy = pd.concat([df_copy, dummies], axis=1)
        df_copy.drop(column, axis=1, inplace=True)
    
    return df_copy

encoding_mappings = {}

# aplicación de codificación a customers_df
customers_df, state_mapping = apply_label_encoding(customers_df, 'customer_state', return_mapping=True)
encoding_mappings['customer_state'] = state_mapping

# Codificación selectiva para city (si tiene muchas categorías)
if customers_df['customer_city'].nunique() <= 15:
    customers_df = apply_onehot_encoding(customers_df, 'customer_city')
else:
    customers_df, city_mapping = apply_label_encoding(customers_df, 'customer_city', return_mapping=True)
    encoding_mappings['customer_city'] = city_mapping

# Codificación para sellers_df
sellers_df, seller_state_mapping = apply_label_encoding(sellers_df, 'seller_state', return_mapping=True)
encoding_mappings['seller_state'] = seller_state_mapping

# Codificación para sellers_city (evaluar la cantidad de categorías)
if sellers_df['seller_city'].nunique() <= 15:
    sellers_df = apply_onehot_encoding(sellers_df, 'seller_city')
else:
    sellers_df = apply_label_encoding(sellers_df, 'seller_city')

# Codificación para orders_df - usar one-hot para order_status (pocas categorías)
orders_df = apply_onehot_encoding(orders_df, 'order_status')
orders_df = apply_onehot_encoding(orders_df, 'order_status_detailed')

# Codificación para order_payments_df
order_payments_df = apply_onehot_encoding(order_payments_df, 'payment_type')

# Codificación para categorías de productos
# Primero unir con la traducción
products_with_categories = pd.merge(products_df, category_translation_df, on='product_category_name', how='left')
products_with_categories['product_category_name_english'] = products_with_categories['product_category_name_english'].fillna('unknown')

# Aplicar label encoding a la categoría en inglés
products_with_categories, category_mapping = apply_label_encoding(
    products_with_categories, 'product_category_name_english', return_mapping=True)
encoding_mappings['product_category_name_english'] = category_mapping

def replace_ids(dataframes, id_columns, mapping_dicts, additional_columns=None):
    if additional_columns is None:
        additional_columns = []

    for df in dataframes:
        for column in id_columns.union(additional_columns):
            if column in df.columns:
                # Crear un mapeo único para la columna si no existe
                if column not in mapping_dicts:
                    unique_ids = df[column].unique()
                    mapping_dicts[column] = {old_id: new_id for new_id, old_id in enumerate(unique_ids, start=1)}
                
                # Reemplazar los IDs largos con los IDs simplificados
                df[column] = df[column].map(mapping_dicts[column])
    return dataframes

# Identificar columnas que terminan en '_id' en los DataFrames
id_columns = set()
for df in df_lists:
    id_columns.update([col for col in df.columns if col.endswith('_id')])

# Agregar columnas adicionales que quieras modificar
additional_columns = {'customer_zip_code_prefix', 'customer_city', 'customer_state'}

# Crear un diccionario para almacenar los mapeos de IDs
id_mappings = {}

# Aplicar la función a los DataFrames
df_lists = replace_ids(df_lists, id_columns, id_mappings, additional_columns)

# Guardar los mapeos de IDs en un archivo JSON

def ids(df, id_column, prefix, start_index=1):

    # Verificar si la columna existe
    if id_column not in df.columns:
        return df, {}
    
    # Obtener IDs únicos
    unique_ids = df[id_column].unique()
    
    # Crear mapeo
    id_mapping = {old_id: f"{prefix}{i}" for i, old_id in enumerate(unique_ids, start_index)}
    
    # Reemplazar directamente los IDs originales con los legibles
    df_copy = df.copy()
    df_copy[id_column] = df_copy[id_column].map(id_mapping)
    
    return df_copy, id_mapping


### **Eliminar Columnas Innecesarias**

In [6]:
# Eliminar columnas innecesarias
if 'customer_unique_id' in customers_df.columns:
    customers_df = customers_df.drop(columns=['customer_unique_id'])
    print("Columna 'customer_unique_id' eliminada del DataFrame de clientes.")

# Generar IDs legibles para las entidades principales
customers_df, customer_id_mapping = ids(customers_df, 'customer_id', 'CUS_')

sellers_df, seller_id_mapping = ids(sellers_df, 'seller_id', 'SEL_')

products_with_categories, product_id_mapping = ids(products_with_categories, 'product_id', 'PRO_')

orders_df, order_id_mapping = ids(orders_df, 'order_id', 'ORD_')

# Actualiza customer_id en orders_df
if 'customer_id' in orders_df.columns:
    orders_df['customer_id'] = orders_df['customer_id'].map(lambda x: customer_id_mapping.get(x, x))

# Actualiza claves foráneas en order_items_df
if 'order_id' in order_items_df.columns:
    order_items_df['order_id'] = order_items_df['order_id'].map(lambda x: order_id_mapping.get(x, x))
if 'product_id' in order_items_df.columns:
    order_items_df['product_id'] = order_items_df['product_id'].map(lambda x: product_id_mapping.get(x, x))
if 'seller_id' in order_items_df.columns:
    order_items_df['seller_id'] = order_items_df['seller_id'].map(lambda x: seller_id_mapping.get(x, x))

# Actualiza claves foráneas en order_payments_df
if 'order_id' in order_payments_df.columns:
    order_payments_df['order_id'] = order_payments_df['order_id'].map(lambda x: order_id_mapping.get(x, x))

# Manejo de order_reviews_df
if 'order_id' in order_reviews_df.columns:
    order_reviews_df['order_id'] = order_reviews_df['order_id'].astype(str)
    mask_nan = order_reviews_df['order_id'].isin(['nan', 'None', '']) | order_reviews_df['order_id'].str.lower().isin(['nan', 'none'])
    order_reviews_df.loc[mask_nan, 'order_id'] = 'unknown'
    
    mask_numeric = ~mask_nan & order_reviews_df['order_id'].str.contains('\.')
    if mask_numeric.any():
        safe_numeric_mask = mask_numeric & order_reviews_df['order_id'].str.replace('.', '', regex=False).str.isdigit()
        if safe_numeric_mask.any():
            numeric_ids = order_reviews_df.loc[safe_numeric_mask, 'order_id']
            order_reviews_df.loc[safe_numeric_mask, 'order_id'] = numeric_ids.astype(float).fillna(0).astype(int).astype(str)
    string_order_mapping = {str(k): v for k, v in order_id_mapping.items()}

    order_reviews_df['order_id'] = order_reviews_df['order_id'].map(
        lambda x: string_order_mapping.get(x, f"OR_{x}" if x != 'unknown' else "OR_unknown")
    )
    

if 'review_id' in order_reviews_df.columns:
    order_reviews_df, _ = ids(order_reviews_df, 'review_id', 'RV_')

with open("encoding_mappings.json", "w", encoding="utf-8") as f:
    json.dump(encoding_mappings, f, indent=4, ensure_ascii=False)

<>:37: SyntaxWarning: invalid escape sequence '\.'
<>:37: SyntaxWarning: invalid escape sequence '\.'
C:\Users\drkfa\AppData\Local\Temp\ipykernel_22012\3613412248.py:37: SyntaxWarning: invalid escape sequence '\.'
  mask_numeric = ~mask_nan & order_reviews_df['order_id'].str.contains('\.')


Columna 'customer_unique_id' eliminada del DataFrame de clientes.


### **Guardar datos procesados**

In [7]:
#processed data
processed_path = data_path + 'processed/'
products_with_categories.to_csv(processed_path + 'processed_products.csv', index=False)
orders_df.to_csv(processed_path + 'processed_orders.csv', index=False)
customers_df.to_csv(processed_path + 'processed_customers.csv', index=False)
sellers_df.to_csv(processed_path + 'processed_sellers.csv', index=False)
order_payments_df.to_csv(processed_path + 'processed_payments.csv', index=False)
order_reviews_df.to_csv(processed_path + 'processed_reviews.csv', index=False)
order_items_df.to_csv(processed_path + 'processed_order_items.csv', index=False)

### **Creación de Datasets Finales**

In [9]:
data_path = 'data/processed/'

customers_df = pd.read_csv(data_path + 'processed_customers.csv')
order_items_df = pd.read_csv(data_path + 'processed_order_items.csv')
orders_df = pd.read_csv(data_path + 'processed_orders.csv')
payments_df = pd.read_csv(data_path + 'processed_payments.csv')
products_df = pd.read_csv(data_path + 'processed_products.csv')
reviews_df = pd.read_csv(data_path + 'processed_reviews.csv')
sellers_df = pd.read_csv(data_path + 'processed_sellers.csv')

reviews_df = reviews_df.drop_duplicates()

# Convertir todos los IDs a string
order_items_df["order_id"] = "ORD_" + order_items_df["order_id"].astype(str)
order_items_df["product_id"] = "PRO_" + order_items_df["product_id"].astype(str)
order_items_df["seller_id"] = "SEL_" + order_items_df["seller_id"].astype(str)

payments_df["order_id"] = payments_df["order_id"].astype(str)
reviews_df["order_id"] = reviews_df["order_id"].astype(str)

#### 1. `df_delivery`

In [ ]:
df_delivery = orders_df.merge(order_items_df, on='order_id', how='left') \
    .merge(customers_df, on='customer_id', how='left') \
    .merge(sellers_df, on='seller_id', how='left')\
    .merge(products_df[['product_id', 'product_category_name']], on='product_id', how='left')

df_delivery["actual_delivery_days"] = (
    pd.to_datetime(df_delivery["order_delivered_customer_date"]) -
    pd.to_datetime(df_delivery["order_purchase_timestamp"])
).dt.days

df_delivery["estimated_delivery_days"] = (
    pd.to_datetime(df_delivery["order_estimated_delivery_date"]) -
    pd.to_datetime(df_delivery["order_purchase_timestamp"])
).dt.days

df_delivery["delivered_late"] = df_delivery["actual_delivery_days"] > df_delivery["estimated_delivery_days"]

# DROP de las columnas innecesarias o redundantes
df_delivery = df_delivery.drop(columns=[
    'order_status_canceled', 'order_status_created', 'order_status_unavailable',
    'order_delivered_carrier_date', 'shipping_limit_date',
    'customer_zip_code_prefix', 'seller_zip_code_prefix'
])

,order_id,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date,order_status_approved,order_status_delivered,order_status_invoiced,order_status_processing,order_status_shipped,order_status_detailed_approved,order_status_detailed_canceled,order_status_detailed_delivered,order_status_detailed_invoiced,order_status_detailed_pending_approval,order_status_detailed_processing,order_status_detailed_shipped,order_status_detailed_unavailable,order_item_id,product_id,seller_id,price,freight_value,customer_city,customer_state,seller_city,seller_state,product_category_name,actual_delivery_days,estimated_delivery_days,delivered_late
0,ORD_1,CUS_70297,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13,2017-10-18 00:00:00,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_1,SEL_1,58.90,13.29,3597,25,101.0,22.0,perfumaria,8.0,15,False
1,ORD_2,CUS_77028,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45,2018-08-13 00:00:00,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_2,SEL_2,239.90,19.93,418,4,342.0,22.0,artes,13.0,19,False
2,ORD_3,CUS_555,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29,2018-09-04 00:00:00,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_3,SEL_3,199.00,17.87,4050,8,450.0,16.0,esporte_lazer,9.0,26,False
3,ORD_4,CUS_61082,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42,2017-12-15 00:00:00,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_4,SEL_4,12.99,12.79,3473,19,517.0,22.0,bebes,13.0,26,False
4,ORD_5,CUS_67264,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02,2018-02-26 00:00:00,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_5,SEL_5,199.90,18.14,3375,25,80.0,22.0,utilidades_domesticas,2.0,12,False


#### 2. `df_order_value`

In [ ]:
df_order_value = orders_df.merge(order_items_df, on='order_id', how='left') \
    .merge(products_df, on='product_id', how='left') \
    .merge(payments_df.groupby("order_id").agg({'payment_value': 'sum'}).reset_index(), on='order_id', how='left')\
    .merge(reviews_df[['order_id', 'review_score']], on='order_id', how='left')

# Drop columnas no útiles
df_order_value = df_order_value.drop(columns=[
    'order_status_canceled', 'order_status_unavailable',
    'shipping_limit_date', 'order_approved_at', 'product_category_name'
])

df_order_value = df_order_value.rename(columns={'product_category_name_english': 'product_category_name'})

,order_id,customer_id,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_status_approved,order_status_created,order_status_delivered,order_status_invoiced,order_status_processing,order_status_shipped,order_status_detailed_approved,order_status_detailed_canceled,order_status_detailed_delivered,order_status_detailed_invoiced,order_status_detailed_pending_approval,order_status_detailed_processing,order_status_detailed_shipped,order_status_detailed_unavailable,order_item_id,product_id,seller_id,price,freight_value,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name,payment_value,review_score
0,ORD_1,CUS_70297,2017-10-02 10:56:33,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,False,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_1,SEL_1,58.90,13.29,40.0,287.0,1.0,225.0,16.0,10.0,14.0,59.0,38.71,NaN
1,ORD_2,CUS_77028,2018-07-24 20:41:37,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,False,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_2,SEL_2,239.90,19.93,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,2.0,141.46,NaN
2,ORD_3,CUS_555,2018-08-08 08:38:49,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,False,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_3,SEL_3,199.00,17.87,46.0,250.0,1.0,154.0,18.0,9.0,15.0,65.0,179.12,NaN
3,ORD_4,CUS_61082,2017-11-18 19:28:06,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,False,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_4,SEL_4,12.99,12.79,27.0,261.0,1.0,371.0,26.0,4.0,26.0,6.0,72.20,NaN
4,ORD_5,CUS_67264,2018-02-13 21:18:39,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,False,False,True,False,False,False,False,False,True,False,False,False,False,False,1.0,PRO_5,SEL_5,199.90,18.14,37.0,402.0,4.0,625.0,20.0,17.0,13.0,49.0,28.62,NaN


#### 3. `df_payments`

In [ ]:
df_payments = payments_df.merge(orders_df, on='order_id', how='left') \
    .merge(customers_df, on='customer_id', how='left')


# Drop detalles no útiles
df_payments = df_payments.drop(columns=[
    'payment_sequential', 'payment_installments', 'customer_zip_code_prefix'
])


,order_id,payment_value,payment_type_boleto,payment_type_credit_card,payment_type_debit_card,payment_type_not_defined,payment_type_voucher,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_status_approved,order_status_canceled,order_status_created,order_status_delivered,order_status_invoiced,order_status_processing,order_status_shipped,order_status_unavailable,order_status_detailed_approved,order_status_detailed_canceled,order_status_detailed_delivered,order_status_detailed_invoiced,order_status_detailed_pending_approval,order_status_detailed_processing,order_status_detailed_shipped,order_status_detailed_unavailable,customer_city,customer_state
0,ORD_75269,99.33,False,True,False,False,False,CUS_49653,2018-04-25 22:01:49,2018-04-25 22:15:09,2018-05-02 15:20:00,2018-05-09 17:36:51,2018-05-22 00:00:00,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,3839,10
1,ORD_98161,24.39,False,True,False,False,False,CUS_75027,2018-06-26 11:01:38,2018-06-26 11:18:58,2018-06-28 14:18:00,2018-06-29 20:32:09,2018-07-16 00:00:00,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,3597,25
2,ORD_43930,65.71,False,True,False,False,False,CUS_26281,2017-12-12 11:19:55,2017-12-14 09:52:34,2017-12-15 20:13:22,2017-12-18 17:24:41,2018-01-04 00:00:00,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,3597,25
3,ORD_64933,107.78,False,True,False,False,False,CUS_77779,2017-12-06 12:04:06,2017-12-06 12:13:20,2017-12-07 20:28:28,2017-12-21 01:35:51,2018-01-04 00:00:00,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,2005,10
4,ORD_12703,128.45,False,True,False,False,False,CUS_13913,2018-05-21 13:59:17,2018-05-21 16:14:41,2018-05-22 11:46:00,2018-06-01 21:44:53,2018-06-13 00:00:00,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,1021,25


#### 4. `df_customers_segmented`

In [ ]:
# Agrupar info por cliente
df_customers_segmented = orders_df.merge(order_items_df, on='order_id', how='left') \
    .groupby('customer_id').agg({
        'order_id': 'count',
        'price': 'sum',
        'freight_value': 'sum'
    }).reset_index()

df_customers_segmented.columns = ['customer_id', 'n_orders', 'total_spent', 'total_freight']

# Merge con ubicación
df_customers_segmented = df_customers_segmented.merge(customers_df, on='customer_id', how='left') \
    .drop(columns=['customer_zip_code_prefix'])

,customer_id,n_orders,total_spent,total_freight,customer_city,customer_state
0,CUS_1,1,65.85,13.12,1382,25
1,CUS_10,1,46.49,22.06,453,10
2,CUS_100,1,13.99,7.78,3566,9
3,CUS_1000,1,319.00,24.77,2579,15
4,CUS_10000,1,287.70,40.06,2005,10


#### 5. `df_sellers_ranked`

In [ ]:
df_sellers_ranked = orders_df.merge(order_items_df, on='order_id', how='left') \
    .merge(sellers_df, on='seller_id', how='left') \
    .merge(reviews_df[['order_id', 'review_score']], on='order_id', how='left')

df_sellers_ranked = df_sellers_ranked.groupby('seller_id').agg({
    'order_id': 'count',
    'price': 'sum',
    'freight_value': 'mean',
    'review_score': 'mean'
}).reset_index()

df_sellers_ranked.columns = ['seller_id', 'n_sales', 'total_sales', 'avg_freight', 'avg_review']

,seller_id,n_sales,total_sales,avg_freight,avg_review
0,SEL_1,151,12271.71,19.284305,NaN
1,SEL_10,86,6110.58,15.616395,NaN
2,SEL_100,41,4249.59,13.600732,NaN
3,SEL_1000,7,2708.96,37.512857,NaN
4,SEL_1001,5,479.50,17.522000,NaN


#### 6. `df_logistics_efficiency`

In [ ]:
df_logistics_efficiency = orders_df.merge(order_items_df, on='order_id', how='left') \
    .merge(customers_df, on='customer_id', how='left') \
    .merge(sellers_df, on='seller_id', how='left')

df_logistics_efficiency["delivery_days"] = (
    pd.to_datetime(df_logistics_efficiency["order_delivered_customer_date"]) -
    pd.to_datetime(df_logistics_efficiency["order_purchase_timestamp"])
).dt.days

# Agrupar por origen-destino
df_logistics_efficiency = df_logistics_efficiency.groupby(
    ['customer_state', 'seller_state']
).agg({
    'delivery_days': 'mean',
    'order_id': 'count'
}).reset_index()

df_logistics_efficiency.columns = ['customer_state', 'seller_state', 'avg_delivery_days', 'n_orders']

,customer_state,seller_state,avg_delivery_days,n_orders
0,0,2.0,18.000000,1
1,0,6.0,15.000000,1
2,0,8.0,17.875000,8
3,0,9.0,25.000000,1
4,0,15.0,22.111111,9


#### 7. `df_product_stats`

In [ ]:
df_product_stats = order_items_df.merge(products_df, on='product_id', how='left') \
    .merge(reviews_df[['order_id', 'review_score']], on='order_id', how='left')

df_product_stats = df_product_stats.groupby(['product_id', 'product_category_name']).agg({
    'price': 'mean',
    'freight_value': 'mean',
    'order_id': 'count',
    'review_score': 'mean'
}).reset_index()

# Crear columna auxiliar con el número real
df_product_stats['product_id_num'] = df_product_stats['product_id'].str.extract(r'PRO_(\d+)').astype(int)

# Ordenar por ese número
df_product_stats = df_product_stats.sort_values('product_id_num').drop(columns='product_id_num').reset_index(drop=True)


df_product_stats.columns = ['product_id', 'product_category_name', 'avg_price', 'avg_freight', 'n_sales', 'avg_review']

,product_id,product_category_name,avg_price,avg_freight,n_sales,avg_review
0,PRO_1,perfumaria,59.233333,22.033333,9,NaN
1,PRO_2,artes,239.900000,19.930000,1,NaN
2,PRO_3,esporte_lazer,199.000000,18.606667,3,NaN
3,PRO_4,bebes,12.990000,12.790000,2,NaN
4,PRO_5,utilidades_domesticas,202.400000,27.052500,12,NaN


#### 8. `df_seller_performance_time`

In [ ]:
df_seller_performance_time = orders_df.merge(order_items_df, on='order_id', how='left') \
    .merge(sellers_df, on='seller_id', how='left') \
    .merge(reviews_df[['order_id', 'review_score']], on='order_id', how='left')

df_seller_performance_time['order_month'] = pd.to_datetime(df_seller_performance_time['order_purchase_timestamp']).dt.to_period('M')

df_seller_performance_time = df_seller_performance_time.groupby(['seller_id', 'order_month']).agg({
    'order_id': 'count',
    'price': 'sum',
    'review_score': 'mean'
}).reset_index()

df_seller_performance_time.columns = ['seller_id', 'order_month', 'n_orders', 'revenue', 'avg_review']

,seller_id,order_month,n_orders,revenue,avg_review
0,SEL_1,2016-10,1,55.9,NaN
1,SEL_1,2017-01,1,45.9,NaN
2,SEL_1,2017-02,3,230.7,NaN
3,SEL_1,2017-03,2,154.8,NaN
4,SEL_1,2017-04,3,224.7,NaN


### Preprocesamiento de cada df

In [ ]:
new_dfs = {
    'df_delivery': df_delivery,
    'df_order_value': df_order_value,
    'df_payments': df_payments,
    'df_customers_segmented': df_customers_segmented,
    'df_sellers_ranked': df_sellers_ranked,
    'df_logistics_efficiency': df_logistics_efficiency,
    'df_product_stats': df_product_stats,
    'df_seller_performance_time': df_seller_performance_time
}

# Codificar las columnas booleanas
for df_name, df in new_dfs.items():
    for col in df.select_dtypes(include=[bool]).columns:
        df[col] = df[col].astype(int)

# Eliminar valores duplicados
category_mapping_dict = products_df.drop_duplicates(subset='product_category_name') \
    .set_index('product_category_name')['product_category_name_english'].to_dict()

# Unier el nombre de categorías a inglés
df_product_stats['product_category_name_english'] = df_product_stats['product_category_name'] \
    .map(category_mapping_dict)

df_product_stats = df_product_stats.drop(columns='product_category_name').rename(columns={'product_category_name_english': 'product_category_name'})

Ahora si, tenemos todos los valores codificados a excepción de los ids y las fechas, nuestros **data frames** ya están preparados para la siguiente fase de nuestro proyecto.

Resumen de los **df** creados:
1. `df_delivery` – **Análisis de tiempos de entrega**
2. `df_reviews` – **Clasificación de satisfacción del cliente**
3. `df_order_value` – **Regresión del valor de una orden**
4. `df_payments` – **Análisis de métodos de pago**
5. `df_customers_segmented` – **Segmentación de clientes**
6. `df_sellers_ranked` – **Ranking de vendedores más eficientes**
7. `df_logistics_efficiency` – **Eficiencia logística por ciudad/estado**
8. `df_product_stats` – **para análisis de productos y categoría**
9. `df_seller_performance_time` – **para analizar evolución temporal**